# Lab Assignment 10: Multi-Head Attention, Transformer Encoders & Fine-Tuning BERT

**Objective:**
In Lab 8 you built a single attention mechanism inside an LSTM. In Lab 9 you fine-tuned a pre-trained Transformer for translation. In this lab you will **open the black box**: build *scaled dot-product attention* and *multi-head attention* from scratch, assemble them into a mini Transformer encoder, and then compare it with a pre-trained **BERT** model fine-tuned on the same task.

**Task:** Sentiment classification on the **SST-2** dataset (positive / negative movie-review sentences).

**Learning outcomes**
1. Implement scaled dot-product and multi-head attention and verify them against PyTorch.
2. Build a Transformer encoder block (attention + feed-forward + residual + LayerNorm + positional encoding).
3. Understand why BERT (bidirectional, pre-trained encoder) is different from the decoder/LSTM models in previous labs.
4. Fine-tune BERT and evaluate with Accuracy and F1.
5. Visualise and interpret what individual attention heads learn.

| Part | Topic | Points |
|---|---|---|
| 1 | Setup & Data | 10 |
| 2 | Scaled Dot-Product Attention | 15 |
| 3 | Multi-Head Attention + Encoder Block | 25 |
| 4 | Mini-Transformer trained from scratch | 15 |
| 5 | Fine-tuning BERT | 20 |
| 6 | Attention Head Visualisation | 10 |
| 7 | Comparative Analysis & Reflection | 5 |
| | **Total** | **100** |


## Theory Recap

**Scaled dot-product attention** (Lab 8 used an un-scaled version):

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

* `Q` (queries), `K` (keys), `V` (values) are linear projections of the input.
* Dividing by $\sqrt{d_k}$ stops the dot products from growing large, which would push softmax into regions with tiny gradients.

**Multi-head attention** runs `h` attention operations in parallel, each in a lower-dimensional subspace ($d_k = d_{model}/h$), then concatenates and projects the results:

$$\text{MHA}(X)=\text{Concat}(\text{head}_1,\dots,\text{head}_h)W^O$$

Different heads can specialise (e.g. syntax, neighbouring words, rare-word focus).

**BERT** = a stack of Transformer *encoder* blocks, pre-trained with Masked Language Modelling (MLM) and Next-Sentence Prediction (NSP). The `[CLS]` token's final hidden state is used as the sentence representation for classification.

| Model | Direction | Attention type | Pre-trained? |
|---|---|---|---|
| LSTM + Attention (Lab 8) | Left→right encoder (or BiLSTM) | Cross-attention only | No |
| Marian Transformer (Lab 9) | Encoder-decoder | Self + Cross | Yes |
| **BERT (this lab)** | Encoder only, bidirectional | Self-attention | Yes |


## Part 1: Setup and Data (10 points)

### 1.1 Install & import libraries

In [ ]:
!pip install -q transformers datasets evaluate accelerate bertviz scikit-learn matplotlib seaborn

In [ ]:
# TODO: import torch, torch.nn as nn, math, numpy, matplotlib, seaborn
# TODO: from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# TODO: from datasets import load_dataset
# TODO: set a random seed (torch, numpy) and define `device` (cuda if available)


### 1.2 Load SST-2
Use `load_dataset("glue", "sst2")`. To keep training fast on a Colab T4, use a **subset**: 5,000 training samples and the full validation set (872 samples) as your test set.
Print the dataset structure, 3 random examples, and the class distribution.

In [ ]:
# TODO

### 1.3 Tokenize
Load the `bert-base-uncased` tokenizer. Tokenize with `truncation=True`, `max_length=64`, and pad dynamically (use `DataCollatorWithPadding`).
Print one sentence together with its `input_ids`, the decoded tokens (`convert_ids_to_tokens`) and the `attention_mask`. **Explain in a markdown cell:** what are `[CLS]`, `[SEP]` and `##` word-pieces?

In [ ]:
# TODO

## Part 2: Scaled Dot-Product Attention from Scratch (15 points)

### 2.1 Implement the function
Write `scaled_dot_product_attention(Q, K, V, mask=None)` that returns `(output, attention_weights)`.

Requirements:
* Input shapes: `(batch, heads, seq_len, d_k)`
* Scale by `sqrt(d_k)`
* If `mask` is given (1 = keep, 0 = padding), set masked scores to a very large negative number **before** softmax
* Do **not** use `nn.MultiheadAttention` or `F.scaled_dot_product_attention` in your implementation

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # TODO
    pass


### 2.2 Verify
Create random `Q, K, V` and compare your output with `torch.nn.functional.scaled_dot_product_attention` using `torch.allclose`. Also check that every row of the weights sums to 1 and that padded positions receive ~0 weight.

In [ ]:
# TODO

### 2.3 Short question
Compare this with the attention you wrote in **Lab 8** (Bahdanau / dot-product). In 3-4 lines: what is the difference between *additive* and *scaled dot-product* attention, and why does the scaling factor matter? *(Hint: try d_k = 512 with and without scaling and print the std of the scores and the max softmax weight.)*

In [ ]:
# TODO: experiment showing the effect of scaling

## Part 3: Multi-Head Attention and the Transformer Encoder Block (25 points)

### 3.1 MultiHeadAttention layer
Implement `class MultiHeadAttention(nn.Module)` with:
* `__init__(self, d_model, num_heads)` — assert `d_model % num_heads == 0`; create `W_q, W_k, W_v, W_o` as `nn.Linear`
* `split_heads(x)` : `(B, T, d_model) -> (B, h, T, d_k)`
* `forward(x, mask=None)` : project → split → attention (your function from Part 2) → merge heads → `W_o`
* store the last attention weights in `self.attn_weights` for visualisation later

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # TODO
    def split_heads(self, x):
        # TODO
        pass
    def forward(self, x, mask=None):
        # TODO
        pass


### 3.2 Sanity check
Copy the weights of your layer into `nn.MultiheadAttention(d_model, num_heads, batch_first=True)` (hint: its `in_proj_weight` stacks Q, K, V) and confirm the outputs match (`atol=1e-5`).

In [ ]:
# TODO

### 3.3 Positional Encoding
Attention has no notion of order. Implement sinusoidal positional encoding and plot it as a heatmap (positions × dimensions).

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        # TODO
    def forward(self, x):
        # TODO
        pass

# TODO: heatmap of the encoding matrix


### 3.4 Transformer Encoder Block
Implement `EncoderBlock(d_model, num_heads, d_ff, dropout)`:

`x → MHA → Dropout → Add & LayerNorm → FeedForward (Linear-GELU-Linear) → Dropout → Add & LayerNorm`

Then answer in markdown: **Why are residual connections and LayerNorm needed in deep Transformers?**

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # TODO
    def forward(self, x, mask=None):
        # TODO
        pass


## Part 4: Mini-Transformer Classifier Trained from Scratch (15 points)

### 4.1 Build the model
Create `MiniTransformerClassifier` using: token embedding (use the BERT tokenizer's vocab size) → positional encoding → **2** `EncoderBlock`s (`d_model=128`, `heads=4`, `d_ff=256`) → mean-pool over non-padding tokens → `Linear(d_model, 2)`.
Print the number of trainable parameters.

In [ ]:
# TODO

### 4.2 Train
Write a standard PyTorch training loop (AdamW, lr=1e-3, 5 epochs, batch size 32). Log train loss and validation accuracy per epoch and plot both curves.
Note the padding mask must be built from `attention_mask` and reshaped to `(B, 1, 1, T)`.

In [ ]:
# TODO

### 4.3 Evaluate
Report accuracy, F1 and the confusion matrix on the test set. Record the **total training time**.

In [ ]:
# TODO

## Part 5: Fine-Tuning BERT (20 points)

### 5.1 Load the pre-trained model
Load `AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)`. Print the parameter count and compare it with your mini-Transformer.

In [ ]:
# TODO

### 5.2 Metrics
Define `compute_metrics` that returns **accuracy** and **F1** (use `evaluate` or `sklearn`).

In [ ]:
# TODO

### 5.3 Configure and run training
Use `TrainingArguments`: `learning_rate=2e-5`, `num_train_epochs=3`, `per_device_train_batch_size=32`, `weight_decay=0.01`, `eval_strategy="epoch"`, `fp16=True` (GPU), `report_to="none"`. Train with `Trainer` and record training time.

In [ ]:
# TODO

### 5.4 Ablation: frozen vs. full fine-tuning
Re-load a fresh BERT and **freeze all encoder layers** (`for p in model.bert.parameters(): p.requires_grad = False`) so only the classifier head trains. Train with the same settings (you may use lr=1e-3).
Report a table with *trainable parameters*, *accuracy*, *F1*, and *training time* for: frozen BERT vs. fully fine-tuned BERT.

In [ ]:
# TODO

### 5.5 Inference
Use `pipeline("text-classification", model=..., tokenizer=...)` to classify these sentences, and add **3 of your own** (include one sarcastic and one negated sentence):
```python
tests = [
    "This movie was an absolute masterpiece.",
    "I wasted two hours of my life on this film.",
    "The acting was not bad at all.",
    "The plot was predictable, but the visuals were stunning.",
]
```

In [ ]:
# TODO

## Part 6: Visualising Attention Heads (10 points)

### 6.1 Extract attentions
Load your fine-tuned BERT with `output_attentions=True` and run one sentence. `outputs.attentions` is a tuple of 12 tensors with shape `(1, 12, T, T)` — one per layer.

In [ ]:
# TODO

### 6.2 Heatmaps
Plot a **grid of 4 heads** (e.g. layer 1 vs layer 12, two heads each) as heatmaps with tokens on the axes. Then use `bertviz` (`head_view` or `model_view`) to produce an interactive view.

> In Colab the `bertviz` call must be the last line of the cell.

In [ ]:
# TODO

### 6.3 Interpret
In markdown, answer:
1. Do you find a head that mostly attends to the **next / previous token**? To `[SEP]` or `[CLS]`?
2. How do attention patterns differ between **early** and **late** layers?
3. Compare to your from-scratch model's `attn_weights` (Part 3). Are they as structured? Why / why not?

In [ ]:
# TODO: optional — plot the attention of your mini-Transformer for the same sentence

## Part 7: Comparative Analysis & Reflection (5 points)

Fill in the table with your measured results.

| Model | Params | Pre-trained | Test Acc | Test F1 | Train time |
|---|---|---|---|---|---|
| Mini-Transformer (scratch) | ... | No | ... | ... | ... |
| BERT (frozen, head only) | ... | Yes | ... | ... | ... |
| BERT (full fine-tune) | ... | Yes | ... | ... | ... |

**Answer briefly (3–5 lines each):**
1. Why does BERT outperform the same-architecture model trained from scratch, even though both use self-attention?
2. BERT is *encoder-only* and bidirectional, while the Lab 9 Marian model is *encoder-decoder*. Why is BERT unsuitable for generating translations directly?
3. What is the purpose of multiple heads rather than one big head? Give evidence from Part 6.
4. Compare the three paradigms from Labs 8, 9 and 10: *LSTM + attention*, *pre-trained seq2seq Transformer*, *pre-trained encoder (BERT)*. When would you use each?


## What to Submit
* The completed notebook with all outputs visible.
* The comparison table from Part 7 and your written answers.
* Screenshots (or outputs) of the attention heatmaps from Part 6.

**Bonus (+5):** Fine-tune a smaller model (`distilbert-base-uncased`) and add it to the table, commenting on the speed/accuracy trade-off.